<a href="https://colab.research.google.com/github/ellistechcode-droid/GreenBookAI/blob/main/DEProj_DylanAndLee.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Data Engineering Project
Dylan OBrien and Lee Ellis

In [ ]:
from google.colab import drive

# Connect Colab to the repository stored in Google Drive.
drive.mount("/content/drive")

Mounted at /content/drive


## 1. Project setup and repository paths
The project repository may be stored inside another Google Drive folder. The code below searches Google Drive for the expected repository folder and verifies that it contains the main skeleton directories. If the repository folder has been renamed, update `REPO_FOLDER_NAME`.

In [ ]:
try:
    import pandas as pd
except ImportError as exc:
    raise ImportError(
        "VersionGuard requires pandas. Install pandas before running the notebook."
    ) from exc

try:
    import requests
except ImportError as exc:
    raise ImportError(
        "VersionGuard requires requests. Install requests before running the notebook."
    ) from exc

from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4
import hashlib
import json


def utc_now():
    """Return the current UTC time in a consistent format."""
    return (
        datetime.now(timezone.utc)
        .isoformat(timespec="seconds")
        .replace("+00:00", "Z")
    )


# Update this path so it points directly to the repository in Google Drive.
repo_root = Path(
    "/content/drive/MyDrive/Colab Notebooks/Data Engineering Final Project/ai-de-final-project-dylan-and-lee-main"
)

NOTEBOOK_NAME = "DEProj_DylanAndLee.ipynb"


# Confirm that the selected folder is the correct repository.
required_items = ["README.md", "src", "data", "artifacts"]
missing_items = [
    item
    for item in required_items
    if not (repo_root / item).exists()
]

if missing_items:
    raise FileNotFoundError(
        f"The repository was not found at:\n{repo_root}\n\n"
        f"Missing expected items: {missing_items}\n"
        "Update repo_root so it points to the correct Google Drive folder."
    )


data_sample_dir = repo_root / "data" / "sample"
outputs_dir = repo_root / "artifacts" / "prototype-evidence" / "outputs"
main_notebook_path = repo_root / "src" / NOTEBOOK_NAME

# Defensive checks in case an empty output folder was removed.
data_sample_dir.mkdir(parents=True, exist_ok=True)
outputs_dir.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {repo_root}")
print(f"Expected notebook location: src/{NOTEBOOK_NAME}")
print(f"Sample data directory: {data_sample_dir.relative_to(repo_root)}")
print(f"Evidence output directory: {outputs_dir.relative_to(repo_root)}")

Repository root: /content/drive/MyDrive/Colab Notebooks/Data Engineering Final Project/ai-de-final-project-dylan-and-lee-main
Expected notebook location: src/DEProj_DylanAndLee.ipynb
Sample data directory: data/sample
Evidence output directory: artifacts/prototype-evidence/outputs


## 2. Source Registry

The first VersionGuard corpus contains seven official, version-pinned Pydantic Markdown documents.

The corpus covers three version-sensitive topics shared by Pydantic V1 and V2:

- Models
- Validators
- Configuration

It also includes the official V1-to-V2 migration guide.

The raw GitHub URLs are pinned to exact releases so that repeated pipeline runs use the same source documents. The rendered documentation URLs are retained so that users can open the human-readable source page.

In [ ]:
SOURCES = [
    {
        "document_id": "pydantic-v1-models",
        "title": "Pydantic V1 Models",
        "pydantic_version": "1.10.24",
        "version_group": "v1",
        "topic": "models",
        "source_url": "https://raw.githubusercontent.com/pydantic/pydantic/v1.10.24/docs/usage/models.md",
        "documentation_url": "https://pydantic.dev/docs/validation/1.10/usage/models/",

    },
    {
        "document_id": "pydantic-v1-validators",
        "title": "Pydantic V1 Validators",
        "pydantic_version": "1.10.24",
        "version_group": "v1",
        "topic": "validators",
        "source_url": "https://raw.githubusercontent.com/pydantic/pydantic/v1.10.24/docs/usage/validators.md",
        "documentation_url": "https://pydantic.dev/docs/validation/1.10/usage/models/",

    },
    {
        "document_id": "pydantic-v1-model_config",
        "title": "Pydantic V1 Model Config",
        "pydantic_version": "1.10.24",
        "version_group": "v1",
        "topic": "model_config",
        "source_url": "https://raw.githubusercontent.com/pydantic/pydantic/v1.10.24/docs/usage/model_config.md",
        "documentation_url": "https://pydantic.dev/docs/validation/1.10/usage/models/",

    },
    {
        "document_id": "pydantic-v2-models",
        "title": "Pydantic V2 Models",
        "pydantic_version": "2.13.4",
        "version_group": "v2",
        "topic": "models",
        "source_url": "https://raw.githubusercontent.com/pydantic/pydantic/v2.13.4/docs/concepts/models.md",
        "documentation_url": "https://pydantic.dev/docs/validation/1.10/usage/models/",

    },
    {
        "document_id": "pydantic-v2-validators",
        "title": "Pydantic V2 Validators",
        "pydantic_version": "2.13.4",
        "version_group": "v2",
        "topic": "validators",
        "source_url": "https://raw.githubusercontent.com/pydantic/pydantic/v2.13.4/docs/concepts/validators.md",
        "documentation_url": "https://pydantic.dev/docs/validation/1.10/usage/models/",

    },
    {
        "document_id": "pydantic-V2-config",
        "title": "Pydantic V2 Config",
        "pydantic_version": "2.13.4",
        "version_group": "v2",
        "topic": "config",
        "source_url": "https://raw.githubusercontent.com/pydantic/pydantic/v2.13.4/docs/concepts/config.md",
        "documentation_url": "https://pydantic.dev/docs/validation/1.10/usage/models/",

    },
    {
        "document_id": "pydantic-v2-migration",
        "title": "Pydantic V2 Migration",
        "pydantic_version": "1-to-2",
        "version_group": "migration",
        "topic": "migration",
        "source_url": "https://raw.githubusercontent.com/pydantic/pydantic/v2.13.4/docs/migration.md",
        "documentation_url": "https://pydantic.dev/docs/validation/1.10/usage/models/",

    },
]

sources_df = pd.DataFrame(SOURCES).assign(
    source_meaning="knowledge_asset",
    source_authority="official",
    capture_mode="batch_http_download",
    sensitivity="public",
)

sources_df[["document_id", "version_group", "topic", "source_url"]]

,document_id,version_group,topic,source_url
0,pydantic-v1-models,v1,models,https://raw.githubusercontent.com/pydantic/pyd...
1,pydantic-v1-validators,v1,validators,https://raw.githubusercontent.com/pydantic/pyd...
2,pydantic-v1-model_config,v1,model_config,https://raw.githubusercontent.com/pydantic/pyd...
3,pydantic-v2-models,v2,models,https://raw.githubusercontent.com/pydantic/pyd...
4,pydantic-v2-validators,v2,validators,https://raw.githubusercontent.com/pydantic/pyd...
5,pydantic-V2-config,v2,config,https://raw.githubusercontent.com/pydantic/pyd...
6,pydantic-v2-migration,migration,migration,https://raw.githubusercontent.com/pydantic/pyd...


## 3. Batch Ingestion

Each registered Markdown file is downloaded from its version-pinned GitHub URL.

The ingestion process records:

- The HTTP status
- Whether the download succeeded
- Any download error
- The capture timestamp
- The complete source text
- The number of characters
- A SHA-256 content hash

The content hash provides simple evidence that can later be used to determine whether a source document changed.

In [ ]:
run_started_at_utc = utc_now()

session = requests.Session()
session.headers.update(
    {"User-Agent": "VersionGuard academic data engineering prototype"}
)


def download_source(source):
    """Download one documentation source and record ingestion metadata."""

    row = dict(source)

    # Default values allow failed requests to remain visible in the dataset.
    row.update(
        capture_timestamp_utc=utc_now(),
        http_status=None,
        download_success=False,
        download_error="",
        text="",
        character_count=0,
        content_sha256="",
    )

    try:
        response = session.get(source["source_url"], timeout=30)
        row["http_status"] = int(response.status_code)
        response.raise_for_status()

        text = response.content.decode("utf-8")

        row.update(
            download_success=True,
            text=text,
            character_count=len(text),
            content_sha256=hashlib.sha256(
                text.encode("utf-8")
            ).hexdigest(),
        )

    except Exception as exc:
        response = getattr(exc, "response", None)

        if row["http_status"] is None and response is not None:
            row["http_status"] = int(response.status_code)

        row["download_error"] = f"{type(exc).__name__}: {exc}"

    return row


downloaded_records = [
    download_source(source)
    for source in sources_df.to_dict("records")
]

downloaded_df = pd.DataFrame(downloaded_records)
downloaded_df["record_origin"] = "downloaded_source"

display(
    downloaded_df[
        [
            "document_id",
            "http_status",
            "character_count",
            "download_success",
            "download_error",
        ]
    ]
)

,document_id,http_status,character_count,download_success,download_error
0,pydantic-v1-models,200,23645,True,
1,pydantic-v1-validators,200,5119,True,
2,pydantic-v1-model_config,200,8692,True,
3,pydantic-v2-models,200,59232,True,
4,pydantic-v2-validators,200,30236,True,
5,pydantic-V2-config,200,6282,True,
6,pydantic-v2-migration,200,50005,True,


## 4. Interaction with the Source Data

The following summaries confirm that the notebook is working with real Pydantic documentation.

Only short previews are displayed because the full Markdown documents are stored in the DataFrame and later written to the project data files.

In [ ]:
def compact_preview(text, limit=450):
    """Return a short, single-line preview of a document."""
    cleaned_text = " ".join(str(text).split())
    return cleaned_text[:limit]


fetch_summary = (
    downloaded_df.groupby("version_group", dropna=False)
    .agg(
        registered_sources=("document_id", "count"),
        successful_downloads=("download_success", "sum"),
        failed_downloads=(
            "download_success",
            lambda values: int((~values).sum()),
        ),
        total_characters=("character_count", "sum"),
    )
    .reset_index()
)

fetch_summary["successful_downloads"] = (
    fetch_summary["successful_downloads"].astype(int)
)

downloaded_df["text_snippet"] = downloaded_df["text"].apply(
    lambda text: compact_preview(text, limit=220)
)

print("Download summary by version group:")
display(fetch_summary)

print("Individual downloaded documents:")
display(
    downloaded_df[
        [
            "title",
            "http_status",
            "character_count",
            "download_success",
            "text_snippet",
        ]
    ]
)


# Display one larger preview from each main version group.
preview_requests = [
    ("v1", "One V1 document"),
    ("v2", "One V2 document"),
    ("migration", "Migration guide"),
]

preview_rows = []

for version_group, label in preview_requests:
    matches = downloaded_df[
        (downloaded_df["version_group"] == version_group)
        & downloaded_df["download_success"]
    ]

    if matches.empty:
        preview_rows.append(
            {
                "preview_type": label,
                "document_id": "",
                "preview": "No successful download is available.",
            }
        )
    else:
        row = matches.iloc[0]

        preview_rows.append(
            {
                "preview_type": label,
                "document_id": row["document_id"],
                "preview": compact_preview(row["text"]),
            }
        )

preview_df = pd.DataFrame(preview_rows)

print("Example source previews:")
display(preview_df)

if int(downloaded_df["download_success"].sum()) == 0:
    print(
        "No Pydantic sources were downloaded. "
        "Check the internet connection and rerun the notebook."
    )

Download summary by version group:


,version_group,registered_sources,successful_downloads,failed_downloads,total_characters
0,migration,1,1,0,50005
1,v1,3,3,0,37456
2,v2,3,3,0,95750


Individual downloaded documents:


,title,http_status,character_count,download_success,text_snippet
0,Pydantic V1 Models,200,23645,True,The primary means of defining objects in *pyda...
1,Pydantic V1 Validators,200,5119,True,Custom validation and complex relationships be...
2,Pydantic V1 Model Config,200,8692,True,Behaviour of _pydantic_ can be controlled via ...
3,Pydantic V2 Models,200,59232,True,"??? api ""API Documentation"" [`pydantic.main.Ba..."
4,Pydantic V2 Validators,200,30236,True,In addition to Pydantic's [built-in validation...
5,Pydantic V2 Config,200,6282,True,The behaviour of Pydantic can be controlled vi...
6,Pydantic V2 Migration,200,50005,True,--- description: Migrating from Pydantic V1. -...


Example source previews:


,preview_type,document_id,preview
0,One V1 document,pydantic-v1-models,The primary means of defining objects in *pyda...
1,One V2 document,pydantic-v2-models,"??? api ""API Documentation"" [`pydantic.main.Ba..."
2,Migration guide,pydantic-v2-migration,--- description: Migrating from Pydantic V1. -...


## 5. Controlled Invalid Record, Validation, and Quarantine

A data pipeline should show how it responds when source data is incomplete or invalid.

To make the quarantine path visible, the notebook creates one controlled test record by copying a downloaded document and removing its Pydantic version. The record is clearly labeled as synthetic test data and is not treated as a real source document.

The validation stage performs four checks:

1. Required fields must be present and nonblank.
2. The source download must have succeeded.
3. The document must contain at least 100 characters.
4. Every document ID must be unique.

Records that pass all checks are accepted. Records that fail one or more checks are quarantined for review.

In [ ]:
# Create one controlled bad record to demonstrate quarantine behavior.
synthetic_record = downloaded_df.iloc[0].copy()
synthetic_record["document_id"] = "validation-test-missing-version"
synthetic_record["pydantic_version"] = ""
synthetic_record["record_origin"] = "synthetic_validation_test"

validation_df = pd.concat(
    [
        downloaded_df,
        synthetic_record.to_frame().T,
    ],
    ignore_index=True,
)


REQUIRED_FIELDS = [
    "document_id",
    "title",
    "pydantic_version",
    "version_group",
    "topic",
    "source_url",
    "documentation_url",
    "capture_timestamp_utc",
    "content_sha256",
    "text",
]


# Find every ID that appears more than once.
duplicate_ids = set(
    validation_df.loc[
        validation_df["document_id"].duplicated(keep=False),
        "document_id",
    ]
)


def is_blank(value):
    """Return True when a required value is missing or empty."""
    if value is None or pd.isna(value):
        return True

    return str(value).strip() == ""


def validate_record(row):
    """Return a list of validation errors for one record."""
    errors = []

    # Check 1: required metadata and content.
    for field in REQUIRED_FIELDS:
        if is_blank(row.get(field)):
            errors.append(f"missing {field}")

    # Check 2: successful source download.
    if not bool(row.get("download_success", False)):
        reason = row.get("download_error") or "unknown request failure"
        errors.append(f"download failed: {reason}")

    # Check 3: minimum source length.
    character_count = row.get("character_count")

    if is_blank(character_count):
        character_count = 0

    if int(character_count) < 100:
        errors.append("text has fewer than 100 characters")

    # Check 4: unique document identifier.
    if row.get("document_id") in duplicate_ids:
        errors.append("duplicate document_id")

    return errors


validation_df["validation_errors"] = validation_df.apply(
    validate_record,
    axis=1,
)

validation_df["validation_status"] = validation_df[
    "validation_errors"
].apply(
    lambda errors: "accepted" if len(errors) == 0 else "quarantined"
)

# Convert error lists into readable CSV-friendly text.
validation_df["validation_errors"] = validation_df[
    "validation_errors"
].apply("; ".join)

accepted_df = validation_df[
    validation_df["validation_status"] == "accepted"
].copy()

quarantined_df = validation_df[
    validation_df["validation_status"] == "quarantined"
].copy()

print("Validation results:")
display(
    validation_df[
        [
            "document_id",
            "version_group",
            "record_origin",
            "validation_status",
            "validation_errors",
        ]
    ]
)

print("Quarantined records:")
display(
    quarantined_df[
        [
            "document_id",
            "record_origin",
            "validation_errors",
        ]
    ]
)

Validation results:


,document_id,version_group,record_origin,validation_status,validation_errors
0,pydantic-v1-models,v1,downloaded_source,accepted,
1,pydantic-v1-validators,v1,downloaded_source,accepted,
2,pydantic-v1-model_config,v1,downloaded_source,accepted,
3,pydantic-v2-models,v2,downloaded_source,accepted,
4,pydantic-v2-validators,v2,downloaded_source,accepted,
5,pydantic-V2-config,v2,downloaded_source,accepted,
6,pydantic-v2-migration,migration,downloaded_source,accepted,
7,validation-test-missing-version,v1,synthetic_validation_test,quarantined,missing pydantic_version


Quarantined records:


,document_id,record_origin,validation_errors
7,validation-test-missing-version,synthetic_validation_test,missing pydantic_version


## 6. Save Project Outputs

The notebook writes stable filenames into the existing repository structure. Running the notebook again replaces the previous files with the newest pipeline run.

The outputs include:

- The downloaded source-document dataset
- A compact document inventory
- Accepted records
- Quarantined records
- A JSON ingestion run log

The run log records the source counts, validation rules, output paths, timestamps, and project stage. This provides basic lineage and evidence for investigating future failures.

In [ ]:
# Exclude the synthetic validation record from the genuine source dataset.
real_source_documents = validation_df[
    validation_df["record_origin"] == "downloaded_source"
].copy()


source_documents_path = (
    data_sample_dir / "pydantic_source_documents.csv"
)

inventory_path = outputs_dir / "document_inventory.csv"
accepted_path = outputs_dir / "accepted_documents.csv"
quarantined_path = outputs_dir / "quarantined_documents.csv"
run_log_path = outputs_dir / "ingestion_run_log.json"


inventory_columns = [
    "document_id",
    "title",
    "pydantic_version",
    "version_group",
    "topic",
    "source_url",
    "documentation_url",
    "http_status",
    "capture_timestamp_utc",
    "character_count",
    "text_snippet",
    "content_sha256",
    "validation_status",
    "validation_errors",
]


# Save the complete source records and validation evidence.
real_source_documents.to_csv(
    source_documents_path,
    index=False,
)

real_source_documents[inventory_columns].to_csv(
    inventory_path,
    index=False,
)

accepted_df.to_csv(
    accepted_path,
    index=False,
)

quarantined_df.to_csv(
    quarantined_path,
    index=False,
)


successful_download_count = int(
    downloaded_df["download_success"].sum()
)

failed_download_count = int(
    (~downloaded_df["download_success"]).sum()
)


def relative_output_path(path):
    """Return a readable path relative to the repository root."""
    return path.relative_to(repo_root).as_posix()


run_log = {
    "run_id": f"versionguard-{uuid4()}",
    "project_architecture": "single_notebook_project_entrypoint",
    "main_notebook_path": f"src/{NOTEBOOK_NAME}",
    "implemented_stage": "source_ingestion_validation_quarantine",
    "run_started_at_utc": run_started_at_utc,
    "run_completed_at_utc": utc_now(),
    "registered_source_count": int(len(sources_df)),
    "successful_download_count": successful_download_count,
    "failed_download_count": failed_download_count,
    "accepted_record_count": int(len(accepted_df)),
    "quarantined_record_count": int(len(quarantined_df)),
    "input_source_type": (
        "Official version-pinned Pydantic Markdown files "
        "downloaded from GitHub raw URLs"
    ),
    "validation_rule_summary": [
        "required fields must be present and not blank",
        "source download must succeed",
        "text must contain at least 100 characters",
        "document_id must not be duplicated",
    ],
    "output_paths": {
        "source_documents": relative_output_path(
            source_documents_path
        ),
        "document_inventory": relative_output_path(
            inventory_path
        ),
        "accepted_documents": relative_output_path(
            accepted_path
        ),
        "quarantined_documents": relative_output_path(
            quarantined_path
        ),
        "ingestion_run_log": relative_output_path(
            run_log_path
        ),
    },
}


with run_log_path.open("w", encoding="utf-8") as file:
    json.dump(run_log, file, indent=2)


print("Saved project outputs:")

for path in [
    source_documents_path,
    inventory_path,
    accepted_path,
    quarantined_path,
    run_log_path,
]:
    print(f"- {relative_output_path(path)}")

Saved project outputs:
- data/sample/pydantic_source_documents.csv
- artifacts/prototype-evidence/outputs/document_inventory.csv
- artifacts/prototype-evidence/outputs/accepted_documents.csv
- artifacts/prototype-evidence/outputs/quarantined_documents.csv
- artifacts/prototype-evidence/outputs/ingestion_run_log.json


## 7. Project Summary

The final summary shows the main counts and output locations from the current pipeline run.

In [ ]:
summary = {
    "registered_sources": int(len(sources_df)),
    "successful_downloads": successful_download_count,
    "failed_downloads": failed_download_count,
    "accepted_records": int(len(accepted_df)),
    "quarantined_records": int(len(quarantined_df)),
    "output_paths": run_log["output_paths"],
}

for key, value in summary.items():
    print(f"{key}: {value}")


if failed_download_count:
    print(
        "\nSome source downloads failed and were quarantined. "
        "Check the recorded errors and rerun the notebook "
        "with a working internet connection."
    )
else:
    print("\nAll registered Pydantic sources downloaded successfully.")


expected_test_record = "validation-test-missing-version"

if expected_test_record in set(quarantined_df["document_id"]):
    print(
        "The controlled missing-version record was correctly quarantined."
    )
else:
    print(
        "Warning: the controlled missing-version record was not quarantined."
    )

registered_sources: 7
successful_downloads: 7
failed_downloads: 0
accepted_records: 7
quarantined_records: 1
output_paths: {'source_documents': 'data/sample/pydantic_source_documents.csv', 'document_inventory': 'artifacts/prototype-evidence/outputs/document_inventory.csv', 'accepted_documents': 'artifacts/prototype-evidence/outputs/accepted_documents.csv', 'quarantined_documents': 'artifacts/prototype-evidence/outputs/quarantined_documents.csv', 'ingestion_run_log': 'artifacts/prototype-evidence/outputs/ingestion_run_log.json'}

All registered Pydantic sources downloaded successfully.
The controlled missing-version record was correctly quarantined.


## 8. Cleaning the Accepted Documentation

The accepted Pydantic documents still contain raw Markdown formatting, repeated blank lines, HTML comments, and other formatting that may not be useful during retrieval.

This step creates a cleaned version of each accepted document while preserving the original source text. Keeping both versions supports reproducibility and makes it possible to compare the processed data with the original downloaded documentation.

The cleaning process removes selected non-content formatting, normalizes whitespace, and records new character counts and hashes for the transformed documents.

In [ ]:
import re


def clean_markdown_text(text):
    """
    Clean raw Markdown while preserving headings,
    code examples, and meaningful documentation content.
    """
    if text is None or pd.isna(text):
        return ""

    cleaned = str(text)

    # Remove HTML comments.
    cleaned = re.sub(
        r"<!--.*?-->",
        "",
        cleaned,
        flags=re.DOTALL,
    )

    # Remove Markdown image syntax.
    cleaned = re.sub(
        r"!\[[^\]]*\]\([^)]+\)",
        "",
        cleaned,
    )

    # Replace tabs with spaces.
    cleaned = cleaned.replace("\t", " ")

    # Remove trailing spaces from each line.
    cleaned = "\n".join(
        line.rstrip()
        for line in cleaned.splitlines()
    )

    # Reduce excessive blank lines.
    cleaned = re.sub(
        r"\n{3,}",
        "\n\n",
        cleaned,
    )

    return cleaned.strip()


# Only accepted real source documents continue into processing.
processing_df = accepted_df[
    accepted_df["record_origin"] == "downloaded_source"
].copy()

processing_df["cleaned_text"] = processing_df["text"].apply(
    clean_markdown_text
)

processing_df["cleaned_character_count"] = processing_df[
    "cleaned_text"
].str.len()

processing_df["cleaned_content_sha256"] = processing_df[
    "cleaned_text"
].apply(
    lambda text: hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()
)

processing_df["characters_removed"] = (
    processing_df["character_count"].astype(int)
    - processing_df["cleaned_character_count"]
)

cleaning_summary = processing_df[
    [
        "document_id",
        "version_group",
        "topic",
        "character_count",
        "cleaned_character_count",
        "characters_removed",
        "cleaned_content_sha256",
    ]
].copy()

print("Cleaning summary:")
display(cleaning_summary)

processing_df["cleaned_text_snippet"] = processing_df[
    "cleaned_text"
].apply(
    lambda text: compact_preview(text, limit=300)
)

print("Cleaned documentation previews:")
display(
    processing_df[
        [
            "document_id",
            "version_group",
            "cleaned_text_snippet",
        ]
    ]
)

Cleaning summary:


,document_id,version_group,topic,character_count,cleaned_character_count,characters_removed,cleaned_content_sha256
0,pydantic-v1-models,v1,models,23645,23595,50,4ade1395a78f01cc298d8a63abe13421b6584eada56c43...
1,pydantic-v1-validators,v1,validators,5119,5118,1,cd23f114ecd8d4b4c51da9b183ae8c6af8018c505f6686...
2,pydantic-v1-model_config,v1,model_config,8692,8688,4,224b9a7d99c0e0e56e9b0bb36af9dd68dea25127b5e0e5...
3,pydantic-v2-models,v2,models,59232,58894,338,9713b6db4f5859b990191aeb5a0dcaf54c3460b92d8e1a...
4,pydantic-v2-validators,v2,validators,30236,30177,59,cc0e9e9a03817d7001c832d1b2bf8408285db5754236ad...
5,pydantic-V2-config,v2,config,6282,6261,21,cc7d0a37cafd7252caada4d01977f973d76415fae084ab...
6,pydantic-v2-migration,migration,migration,50005,49968,37,5deae079a66318a881e00caf0edb702d60a51b7d94355f...


Cleaned documentation previews:


,document_id,version_group,cleaned_text_snippet
0,pydantic-v1-models,v1,The primary means of defining objects in *pyda...
1,pydantic-v1-validators,v1,Custom validation and complex relationships be...
2,pydantic-v1-model_config,v1,Behaviour of _pydantic_ can be controlled via ...
3,pydantic-v2-models,v2,"??? api ""API Documentation"" [`pydantic.main.Ba..."
4,pydantic-v2-validators,v2,In addition to Pydantic's [built-in validation...
5,pydantic-V2-config,v2,The behaviour of Pydantic can be controlled vi...
6,pydantic-v2-migration,migration,--- description: Migrating from Pydantic V1. -...


## 9. Splitting the Documentation into Version-Aware Chunks

The cleaned documents are too large to search as single records. This step divides each document into smaller retrieval chunks while preserving the document title, topic, Pydantic version, section heading, and source information.

The chunking process first separates the Markdown by headings and then divides oversized sections into smaller text windows. A small overlap is included between neighboring chunks so that important context is less likely to be lost at a chunk boundary.

These chunks become the main model-facing knowledge asset used by the retrieval system.

In [ ]:
CHUNK_SIZE = 1200
CHUNK_OVERLAP = 150


def split_markdown_sections(text):
    """
    Split cleaned Markdown into sections using headings.

    Each returned record contains a section heading
    and the text that belongs to that section.
    """
    sections = []
    current_heading = "Document Introduction"
    current_lines = []

    for line in str(text).splitlines():
        stripped_line = line.strip()

        # Detect Markdown headings such as # Models or ## Validators.
        if stripped_line.startswith("#"):
            if current_lines:
                section_text = "\n".join(current_lines).strip()

                if section_text:
                    sections.append(
                        {
                            "section_title": current_heading,
                            "section_text": section_text,
                        }
                    )

            current_heading = stripped_line.lstrip("#").strip()
            current_lines = []
        else:
            current_lines.append(line)

    # Save the final section after the loop ends.
    if current_lines:
        section_text = "\n".join(current_lines).strip()

        if section_text:
            sections.append(
                {
                    "section_title": current_heading,
                    "section_text": section_text,
                }
            )

    return sections


def split_large_section(
    text,
    chunk_size=CHUNK_SIZE,
    overlap=CHUNK_OVERLAP,
):
    """
    Divide a large section into smaller character-based chunks.

    Chunk boundaries are moved to the nearest whitespace when possible.
    """
    text = str(text).strip()

    if not text:
        return []

    if overlap >= chunk_size:
        raise ValueError(
            "CHUNK_OVERLAP must be smaller than CHUNK_SIZE."
        )

    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = min(start + chunk_size, text_length)

        # Avoid cutting through a word when possible.
        if end < text_length:
            whitespace_position = text.rfind(" ", start, end)

            if whitespace_position > start:
                end = whitespace_position

        chunk_text = text[start:end].strip()

        if chunk_text:
            chunks.append(chunk_text)

        if end >= text_length:
            break

        # Move backward slightly to preserve context between chunks.
        next_start = max(end - overlap, start + 1)

        # Move to the next whitespace so the next chunk starts cleanly.
        while (
            next_start < text_length
            and not text[next_start].isspace()
        ):
            next_start += 1

        start = next_start

    return chunks


def create_document_chunks(document_row):
    """
    Create retrieval chunks for one accepted document
    while preserving source and version metadata.
    """
    chunk_records = []
    section_position = 0
    document_chunk_position = 0

    sections = split_markdown_sections(
        document_row["cleaned_text"]
    )

    for section in sections:
        section_position += 1

        section_chunks = split_large_section(
            section["section_text"]
        )

        for section_chunk_position, chunk_text in enumerate(
            section_chunks,
            start=1,
        ):
            document_chunk_position += 1

            chunk_id = (
                f"{document_row['document_id']}"
                f"-chunk-{document_chunk_position:04d}"
            )

            chunk_records.append(
                {
                    "chunk_id": chunk_id,
                    "document_id": document_row["document_id"],
                    "title": document_row["title"],
                    "pydantic_version": document_row[
                        "pydantic_version"
                    ],
                    "version_group": document_row["version_group"],
                    "topic": document_row["topic"],
                    "section_title": section["section_title"],
                    "section_position": section_position,
                    "section_chunk_position": (
                        section_chunk_position
                    ),
                    "document_chunk_position": (
                        document_chunk_position
                    ),
                    "chunk_text": chunk_text,
                    "chunk_character_count": len(chunk_text),
                    "source_url": document_row["source_url"],
                    "documentation_url": document_row[
                        "documentation_url"
                    ],
                    "source_content_sha256": document_row[
                        "content_sha256"
                    ],
                    "cleaned_content_sha256": document_row[
                        "cleaned_content_sha256"
                    ],
                }
            )

    return chunk_records


all_chunk_records = []

for _, document_row in processing_df.iterrows():
    all_chunk_records.extend(
        create_document_chunks(document_row)
    )

chunks_df = pd.DataFrame(all_chunk_records)

# Create a content hash for every retrieval chunk.
chunks_df["chunk_content_sha256"] = chunks_df[
    "chunk_text"
].apply(
    lambda text: hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()
)

print(f"Total retrieval chunks created: {len(chunks_df)}")

chunk_summary = (
    chunks_df.groupby(
        [
            "version_group",
            "document_id",
            "topic",
        ],
        dropna=False,
    )
    .agg(
        chunk_count=("chunk_id", "count"),
        minimum_chunk_length=(
            "chunk_character_count",
            "min",
        ),
        average_chunk_length=(
            "chunk_character_count",
            "mean",
        ),
        maximum_chunk_length=(
            "chunk_character_count",
            "max",
        ),
    )
    .reset_index()
)

chunk_summary["average_chunk_length"] = (
    chunk_summary["average_chunk_length"]
    .round(1)
)

print("Chunk summary by document:")
display(chunk_summary)


chunks_df["chunk_preview"] = chunks_df[
    "chunk_text"
].apply(
    lambda text: compact_preview(text, limit=250)
)

print("Example version-aware chunks:")
display(
    chunks_df[
        [
            "chunk_id",
            "version_group",
            "topic",
            "section_title",
            "chunk_character_count",
            "chunk_preview",
        ]
    ].head(12)
)

Total retrieval chunks created: 303
Chunk summary by document:


,version_group,document_id,topic,chunk_count,minimum_chunk_length,average_chunk_length,maximum_chunk_length
0,migration,pydantic-v2-migration,migration,72,3,707.7,1198
1,v1,pydantic-v1-model_config,model_config,11,210,846.7,1198
2,v1,pydantic-v1-models,models,34,141,696.5,1199
3,v1,pydantic-v1-validators,validators,9,92,563.8,1188
4,v2,pydantic-V2-config,config,14,3,410.0,1153
5,v2,pydantic-v2-models,models,114,3,511.5,1199
6,v2,pydantic-v2-validators,validators,49,3,622.2,1200


Example version-aware chunks:


,chunk_id,version_group,topic,section_title,chunk_character_count,chunk_preview
0,pydantic-v1-models-chunk-0001,v1,models,Document Introduction,1106,The primary means of defining objects in *pyda...
1,pydantic-v1-models-chunk-0002,v1,models,Basic model usage,1199,```py from pydantic import BaseModel class Use...
2,pydantic-v1-models-chunk-0003,v1,models,Basic model usage,561,'123' has been cast to an int as per the field...
3,pydantic-v1-models-chunk-0004,v1,models,Model properties,1197,The example above only shows the tip of the ic...
4,pydantic-v1-models-chunk-0005,v1,models,Model properties,518,cf. [schema](schema.md) `schema_json()` : retu...
5,pydantic-v1-models-chunk-0006,v1,models,Recursive Models,253,More complex hierarchical data structures can ...
6,pydantic-v1-models-chunk-0007,v1,models,ORM Mode (aka Arbitrary Class Instances),395,Pydantic models can be created from arbitrary ...
7,pydantic-v1-models-chunk-0008,v1,models,Reserved names,354,You may want to name a Column after a reserved...
8,pydantic-v1-models-chunk-0009,v1,models,Recursive ORM models,234,ORM instances will be parsed with `from_orm` r...
9,pydantic-v1-models-chunk-0010,v1,models,Data binding,897,Arbitrary classes are processed by *pydantic* ...


## 10. Validating and Quarantining Retrieval Chunks

The retrieval chunks must be validated before they are used by the search system.

This step checks that each chunk has the required metadata, contains enough text to be useful, uses a valid version group, has a unique chunk identifier, and still points to an accepted source document.

Chunks that fail validation are placed in quarantine instead of entering the model-facing retrieval asset.

In [ ]:
VALID_VERSION_GROUPS = {
    "v1",
    "v2",
    "migration",
}

REQUIRED_CHUNK_FIELDS = [
    "chunk_id",
    "document_id",
    "title",
    "pydantic_version",
    "version_group",
    "topic",
    "section_title",
    "chunk_text",
    "chunk_character_count",
    "source_url",
    "source_content_sha256",
    "chunk_content_sha256",
]


# Find chunk IDs that appear more than once.
duplicate_chunk_ids = set(
    chunks_df.loc[
        chunks_df["chunk_id"].duplicated(keep=False),
        "chunk_id",
    ]
)

# Accepted source IDs are used to confirm chunk lineage.
accepted_document_ids = set(
    processing_df["document_id"]
)


def validate_chunk(row):
    """
    Return validation errors for one retrieval chunk.
    """
    errors = []

    # Check 1: required fields must be present.
    for field in REQUIRED_CHUNK_FIELDS:
        if is_blank(row.get(field)):
            errors.append(f"missing {field}")

    # Check 2: chunk identifiers must be unique.
    if row.get("chunk_id") in duplicate_chunk_ids:
        errors.append("duplicate chunk_id")

    # Check 3: chunk must reference an accepted document.
    if row.get("document_id") not in accepted_document_ids:
        errors.append(
            "document_id does not reference an accepted document"
        )

    # Check 4: version group must be recognized.
    if row.get("version_group") not in VALID_VERSION_GROUPS:
        errors.append("invalid version_group")

    # Check 5: chunk must contain enough useful text.
    character_count = row.get("chunk_character_count")

    if is_blank(character_count):
        character_count = 0

    if int(character_count) < 80:
        errors.append("chunk has fewer than 80 characters")

    # Check 6: chunk should not exceed the configured size.
    if int(character_count) > CHUNK_SIZE:
        errors.append(
            f"chunk exceeds configured size of {CHUNK_SIZE}"
        )

    # Check 7: confirm that the chunk hash matches its text.
    chunk_text = str(row.get("chunk_text", ""))

    expected_hash = hashlib.sha256(
        chunk_text.encode("utf-8")
    ).hexdigest()

    if row.get("chunk_content_sha256") != expected_hash:
        errors.append(
            "chunk_content_sha256 does not match chunk_text"
        )

    return errors


chunk_validation_df = chunks_df.copy()

chunk_validation_df["validation_errors"] = (
    chunk_validation_df.apply(
        validate_chunk,
        axis=1,
    )
)

chunk_validation_df["validation_status"] = (
    chunk_validation_df["validation_errors"].apply(
        lambda errors: (
            "accepted"
            if len(errors) == 0
            else "quarantined"
        )
    )
)

chunk_validation_df["validation_errors"] = (
    chunk_validation_df["validation_errors"].apply(
        "; ".join
    )
)

accepted_chunks_df = chunk_validation_df[
    chunk_validation_df["validation_status"] == "accepted"
].copy()

quarantined_chunks_df = chunk_validation_df[
    chunk_validation_df["validation_status"] == "quarantined"
].copy()


print("Chunk validation summary:")

chunk_validation_summary = (
    chunk_validation_df.groupby(
        [
            "version_group",
            "validation_status",
        ],
        dropna=False,
    )
    .agg(
        chunk_count=("chunk_id", "count")
    )
    .reset_index()
)

display(chunk_validation_summary)


print("Accepted retrieval chunks:")
display(
    accepted_chunks_df[
        [
            "chunk_id",
            "document_id",
            "version_group",
            "topic",
            "section_title",
            "chunk_character_count",
        ]
    ].head(12)
)


print("Quarantined retrieval chunks:")

if quarantined_chunks_df.empty:
    print(
        "No retrieval chunks failed validation."
    )
else:
    display(
        quarantined_chunks_df[
            [
                "chunk_id",
                "document_id",
                "version_group",
                "validation_errors",
            ]
        ]
    )


if accepted_chunks_df.empty:
    raise RuntimeError(
        "No retrieval chunks passed validation. "
        "Review the chunking and validation rules."
    )

Chunk validation summary:


,version_group,validation_status,chunk_count
0,migration,accepted,62
1,migration,quarantined,10
2,v1,accepted,54
3,v2,accepted,132
4,v2,quarantined,45


Accepted retrieval chunks:


,chunk_id,document_id,version_group,topic,section_title,chunk_character_count
0,pydantic-v1-models-chunk-0001,pydantic-v1-models,v1,models,Document Introduction,1106
1,pydantic-v1-models-chunk-0002,pydantic-v1-models,v1,models,Basic model usage,1199
2,pydantic-v1-models-chunk-0003,pydantic-v1-models,v1,models,Basic model usage,561
3,pydantic-v1-models-chunk-0004,pydantic-v1-models,v1,models,Model properties,1197
4,pydantic-v1-models-chunk-0005,pydantic-v1-models,v1,models,Model properties,518
5,pydantic-v1-models-chunk-0006,pydantic-v1-models,v1,models,Recursive Models,253
6,pydantic-v1-models-chunk-0007,pydantic-v1-models,v1,models,ORM Mode (aka Arbitrary Class Instances),395
7,pydantic-v1-models-chunk-0008,pydantic-v1-models,v1,models,Reserved names,354
8,pydantic-v1-models-chunk-0009,pydantic-v1-models,v1,models,Recursive ORM models,234
9,pydantic-v1-models-chunk-0010,pydantic-v1-models,v1,models,Data binding,897


Quarantined retrieval chunks:


,chunk_id,document_id,version_group,validation_errors
86,pydantic-v2-models-chunk-0033,pydantic-v2-models,v2,chunk has fewer than 80 characters
87,pydantic-v2-models-chunk-0034,pydantic-v2-models,v2,chunk has fewer than 80 characters
89,pydantic-v2-models-chunk-0036,pydantic-v2-models,v2,chunk has fewer than 80 characters
91,pydantic-v2-models-chunk-0038,pydantic-v2-models,v2,chunk has fewer than 80 characters
92,pydantic-v2-models-chunk-0039,pydantic-v2-models,v2,chunk has fewer than 80 characters
93,pydantic-v2-models-chunk-0040,pydantic-v2-models,v2,chunk has fewer than 80 characters
95,pydantic-v2-models-chunk-0042,pydantic-v2-models,v2,chunk has fewer than 80 characters
96,pydantic-v2-models-chunk-0043,pydantic-v2-models,v2,chunk has fewer than 80 characters
99,pydantic-v2-models-chunk-0046,pydantic-v2-models,v2,chunk has fewer than 80 characters
100,pydantic-v2-models-chunk-0047,pydantic-v2-models,v2,chunk has fewer than 80 characters


## 11. Saving the Versioned Retrieval Asset

The validated retrieval chunks become the primary model-facing knowledge asset for VersionGuard.

This step saves the accepted retrieval chunks together with an asset manifest describing the asset version, creation time, record counts, chunking configuration, and content hash.

The manifest provides reproducibility and lineage by documenting exactly which retrieval asset was produced during this pipeline run.

In [ ]:
from pathlib import Path

# Create filenames for the model-facing retrieval asset.
retrieval_chunks_path = (
    outputs_dir / "retrieval_chunks_asset_v1.csv"
)

retrieval_manifest_path = (
    outputs_dir / "retrieval_chunks_asset_v1_manifest.json"
)

retrieval_index_version = "1.0.0"

# Save the accepted retrieval chunks.
accepted_chunks_df.to_csv(
    retrieval_chunks_path,
    index=False,
)

# Compute one hash representing the complete retrieval asset.
asset_hash = hashlib.sha256(
    accepted_chunks_df.to_json(
        orient="records"
    ).encode("utf-8")
).hexdigest()

retrieval_manifest = {
    "asset_name": "VersionGuard Retrieval Chunks",
    "asset_version": retrieval_index_version,
    "created_at_utc": utc_now(),
    "record_count": int(len(accepted_chunks_df)),
    "source_document_count": int(
        accepted_chunks_df["document_id"].nunique()
    ),
    "schema_version": "1.0",
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "asset_sha256": asset_hash,
    "source_dataset": relative_output_path(
        accepted_path
    ),
    "retrieval_asset": relative_output_path(
        retrieval_chunks_path
    ),
}

with retrieval_manifest_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        retrieval_manifest,
        file,
        indent=2,
    )

print("Retrieval asset saved.")

print(
    f"Chunks: {len(accepted_chunks_df)}"
)

print(
    f"Documents: "
    f"{accepted_chunks_df['document_id'].nunique()}"
)

print(
    f"Asset version: "
    f"{retrieval_index_version}"
)

print(
    f"Asset SHA256:\n{asset_hash}"
)

print("\nSaved files:")

print(
    f"- {relative_output_path(retrieval_chunks_path)}"
)

print(
    f"- {relative_output_path(retrieval_manifest_path)}"
)

Retrieval asset saved.
Chunks: 248
Documents: 7
Asset version: 1.0.0
Asset SHA256:
1c0f42c67ac9976e102cc96630fa8936e97bb0ff1117d8e6059e53027fe5fbd9

Saved files:
- artifacts/prototype-evidence/outputs/retrieval_chunks_asset_v1.csv
- artifacts/prototype-evidence/outputs/retrieval_chunks_asset_v1_manifest.json


## 12. Building the Retrieval Index

The validated retrieval chunks are converted into a searchable index that can support developer questions.

VersionGuard uses a TF-IDF (Term Frequency–Inverse Document Frequency) vectorizer to represent each chunk as a numerical feature vector. This lightweight retrieval approach is appropriate for a notebook prototype while demonstrating the same data flow used by larger retrieval-augmented generation (RAG) systems.

The resulting index becomes the model-facing retrieval structure used to identify the most relevant documentation for a user query.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Configure a simple retrieval model.
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=5000,
)

# Build the TF-IDF matrix using validated chunks.
tfidf_matrix = vectorizer.fit_transform(
    accepted_chunks_df["chunk_text"]
)

retrieval_index = {
    "vectorizer": vectorizer,
    "matrix": tfidf_matrix,
    "asset_version": retrieval_index_version,
    "chunk_count": len(accepted_chunks_df),
}

print("Retrieval index created successfully.\n")

print(f"Indexed chunks: {len(accepted_chunks_df)}")
print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"Matrix shape: {tfidf_matrix.shape}")

# Show a few example vocabulary terms.
sample_terms = sorted(
    list(vectorizer.vocabulary_.keys())
)[:30]

print("\nExample indexed terms:")
print(sample_terms)

Retrieval index created successfully.

Indexed chunks: 248
Vocabulary size: 2210
Matrix shape: (248, 2210)

Example indexed terms:
['00', '000', '01', '01t12', '04', '0636', '0695', '0696', '10', '1047', '1055', '11', '11154', '1178', '12', '123', '13', '14', '1690235887', '17', '1700948167', '1946206415', '20', '2020', '2024', '255', '30x', '321', '3994', '4093']


## 13. Naive Retrieval (Version Ignored)

This section implements a simple retrieval function that searches every validated documentation chunk without considering the requested Pydantic version.

Although this approach often returns technically relevant documentation, it can also retrieve information from the wrong software version. These incorrect but plausible responses represent a quiet failure because they may appear correct while recommending APIs that do not exist in the user's project.

The naive retriever provides a baseline for comparison with the version-aware retrieval method implemented later in the notebook.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity


def naive_retrieve(query, top_k=5):
    """
    Search all retrieval chunks without considering
    the requested Pydantic version.
    """

    query_vector = vectorizer.transform([query])

    similarity_scores = cosine_similarity(
        query_vector,
        tfidf_matrix,
    ).flatten()

    ranked_indices = similarity_scores.argsort()[::-1]

    top_indices = ranked_indices[:top_k]

    results = accepted_chunks_df.iloc[top_indices].copy()

    results["similarity_score"] = (
        similarity_scores[top_indices]
    )

    return results.sort_values(
        by="similarity_score",
        ascending=False,
    )


# Demonstration query.
demo_query = "How do I create a validator?"

naive_results = naive_retrieve(
    demo_query,
    top_k=5,
)

print(f"Query: {demo_query}\n")

display(
    naive_results[
        [
            "version_group",
            "topic",
            "document_id",
            "section_title",
            "similarity_score",
        ]
    ]
)

print("\nExample retrieved chunk:\n")

print(
    naive_results.iloc[0]["chunk_text"][:700]
)

Query: How do I create a validator?



,version_group,topic,document_id,section_title,similarity_score
168,v2,validators,pydantic-v2-validators,Document Introduction,0.234212
195,v2,validators,pydantic-v2-validators,Model validators,0.228214
36,v1,validators,pydantic-v1-validators,Pre and per-item validators,0.201929
292,migration,migration,pydantic-v2-migration,Other changes,0.198856
50,v1,model_config,pydantic-v1-model_config,Change behaviour globally,0.182193



Example retrieved chunk:

In addition to Pydantic's [built-in validation capabilities](./fields.md#field-constraints),
you can leverage custom validators at the field and model levels to enforce more complex constraints
and ensure the integrity of your data.

!!! tip
    Want to quickly jump to the relevant validator section?

    <div class="grid cards" markdown>

    *   Field validators

        ---

        * [field *after* validators](#field-after-validator)
        * [field *before* validators](#field-before-validator)
        * [field *plain* validators](#field-plain-validator)
        * [field *wrap* validators](#field-wrap-validator)

    *   Model validators

        ---

        * [model *before* validator


## 14. Version-Aware Retrieval

Unlike the naive retriever, VersionGuard uses the requested Pydantic version to limit the search space before ranking documents.

Filtering by version reduces the likelihood of retrieving documentation from an incompatible software release. This approach helps prevent quiet failures where an answer appears technically correct but recommends APIs that do not exist in the user's environment.

The version-aware retriever uses the same TF-IDF ranking method as the naive retriever while restricting retrieval to the appropriate documentation version.

In [ ]:
def version_aware_retrieve(
    query,
    requested_version,
    top_k=5,
):
    """
    Retrieve documentation using the requested
    Pydantic version.
    """

    valid_versions = {
        "v1",
        "v2",
        "migration",
    }

    if requested_version not in valid_versions:
        raise ValueError(
            f"Unknown version: {requested_version}"
        )

    # Keep only the requested version.
    candidate_chunks = accepted_chunks_df[
        accepted_chunks_df["version_group"]
        == requested_version
    ].copy()

    if candidate_chunks.empty:
        return candidate_chunks

    candidate_matrix = vectorizer.transform(
        candidate_chunks["chunk_text"]
    )

    query_vector = vectorizer.transform([query])

    similarity_scores = cosine_similarity(
        query_vector,
        candidate_matrix,
    ).flatten()

    candidate_chunks["similarity_score"] = (
        similarity_scores
    )

    return candidate_chunks.sort_values(
        by="similarity_score",
        ascending=False,
    ).head(top_k)


demo_query = "How do I create a validator?"

requested_version = "v1"

aware_results = version_aware_retrieve(
    demo_query,
    requested_version,
)

print(f"Query: {demo_query}")
print(f"Requested version: {requested_version}\n")

display(
    aware_results[
        [
            "version_group",
            "topic",
            "document_id",
            "section_title",
            "similarity_score",
        ]
    ]
)

print("\nExample retrieved chunk:\n")

print(
    aware_results.iloc[0]["chunk_text"][:700]
)

Query: How do I create a validator?
Requested version: v1



,version_group,topic,document_id,section_title,similarity_score
36,v1,validators,pydantic-v1-validators,Pre and per-item validators,0.201929
50,v1,model_config,pydantic-v1-model_config,Change behaviour globally,0.182193
37,v1,validators,pydantic-v1-validators,Subclass Validators and `each_item`,0.172469
41,v1,validators,pydantic-v1-validators,Field Checks,0.162607
14,v1,models,pydantic-v1-models,Creating models without validation,0.132486



Example retrieved chunk:

Validators can do a few more complex things:

{!.tmp_examples/validators_pre_item.md!}

A few more things to note:

* a single validator can be applied to multiple fields by passing it multiple field names
* a single validator can also be called on *all* fields by passing the special value `'*'`
* the keyword argument `pre` will cause the validator to be called prior to other validation
* passing `each_item=True` will result in the validator being applied to individual values
  (e.g. of `List`, `Dict`, `Set`, etc.), rather than the whole object


## 15. Creating the Version-Aware Prompt Template

The retrieval system now returns documentation from the requested Pydantic version. This section creates a reusable prompt template that converts the retrieved chunks into structured context for an assistant response.

The prompt explicitly tells the assistant which Pydantic version the user requested and instructs it not to mix APIs from other versions. The template is saved as a versioned model-facing asset so future runs can identify exactly which prompt instructions were used.

In [ ]:
PROMPT_TEMPLATE_VERSION = "1.0.0"

PROMPT_TEMPLATE = """
You are VersionGuard, a developer documentation assistant.

Requested Pydantic version: {requested_version}

Instructions:
- Answer using only the supplied documentation context.
- Do not mix APIs from different Pydantic versions.
- Clearly state which Pydantic version the answer applies to.
- If the context does not contain enough information, say so.
- Mention the supporting chunk IDs in the answer.

Documentation context:
{context}

User question:
{query}
""".strip()


prompt_template_path = (
    outputs_dir / "prompt_template_v1.txt"
)

prompt_manifest_path = (
    outputs_dir / "prompt_template_v1_manifest.json"
)


with prompt_template_path.open(
    "w",
    encoding="utf-8",
) as file:
    file.write(PROMPT_TEMPLATE)


prompt_template_hash = hashlib.sha256(
    PROMPT_TEMPLATE.encode("utf-8")
).hexdigest()


prompt_manifest = {
    "asset_name": "VersionGuard Prompt Template",
    "asset_version": PROMPT_TEMPLATE_VERSION,
    "created_at_utc": utc_now(),
    "template_sha256": prompt_template_hash,
    "template_path": relative_output_path(
        prompt_template_path
    ),
    "purpose": (
        "Generate version-aware answers from retrieved "
        "Pydantic documentation chunks."
    ),
}


with prompt_manifest_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        prompt_manifest,
        file,
        indent=2,
    )


def build_retrieval_context(results):
    """
    Convert retrieved chunks into labeled prompt context.
    """
    context_blocks = []

    for rank, (_, row) in enumerate(
        results.iterrows(),
        start=1,
    ):
        block = (
            f"[Result {rank}]\n"
            f"Chunk ID: {row['chunk_id']}\n"
            f"Version: {row['version_group']}\n"
            f"Document: {row['title']}\n"
            f"Section: {row['section_title']}\n"
            f"Similarity score: "
            f"{row['similarity_score']:.4f}\n\n"
            f"{row['chunk_text']}"
        )

        context_blocks.append(block)

    return "\n\n---\n\n".join(context_blocks)


def build_version_aware_prompt(
    query,
    requested_version,
    results,
):
    """
    Insert the query, version, and retrieved context
    into the versioned prompt template.
    """
    context = build_retrieval_context(results)

    return PROMPT_TEMPLATE.format(
        requested_version=requested_version,
        context=context,
        query=query,
    )


demo_prompt = build_version_aware_prompt(
    query=demo_query,
    requested_version=requested_version,
    results=aware_results,
)

print("Prompt template saved:")
print(
    f"- {relative_output_path(prompt_template_path)}"
)

print(
    f"- {relative_output_path(prompt_manifest_path)}"
)

print(
    f"\nPrompt version: {PROMPT_TEMPLATE_VERSION}"
)

print(
    f"Prompt SHA256: {prompt_template_hash}"
)

print("\nExample completed prompt:\n")
print(demo_prompt[:2500])

Prompt template saved:
- artifacts/prototype-evidence/outputs/prompt_template_v1.txt
- artifacts/prototype-evidence/outputs/prompt_template_v1_manifest.json

Prompt version: 1.0.0
Prompt SHA256: 2d57e5b3c7839b216ee21f4850f5c0a4584499439046c571d64a4ae7d2bce1cf

Example completed prompt:

You are VersionGuard, a developer documentation assistant.

Requested Pydantic version: v1

Instructions:
- Answer using only the supplied documentation context.
- Do not mix APIs from different Pydantic versions.
- Clearly state which Pydantic version the answer applies to.
- If the context does not contain enough information, say so.
- Mention the supporting chunk IDs in the answer.

Documentation context:
[Result 1]
Chunk ID: pydantic-v1-validators-chunk-0003
Version: v1
Document: Pydantic V1 Validators
Section: Pre and per-item validators
Similarity score: 0.2019

Validators can do a few more complex things:

{!.tmp_examples/validators_pre_item.md!}

A few more things to note:

* a single validator 

## 16. Generating a Grounded Version-Aware Answer

The final step of the retrieval pipeline generates a grounded response using the retrieved documentation chunks.

For this notebook prototype, VersionGuard produces an extractive answer by combining the highest-ranked documentation with version metadata and evidence about the retrieved sources. This approach keeps the project fully reproducible without requiring an external language model while still demonstrating the complete serving pipeline.

The generated response includes the requested version, the retrieved evidence, and the chunk identifiers used to produce the answer.

In [ ]:
def generate_answer(
    query,
    requested_version,
    retrieval_results,
):
    """
    Generate a grounded response using the
    retrieved documentation chunks.
    """

    if retrieval_results.empty:
        return {
            "requested_version": requested_version,
            "query": query,
            "answer": (
                "No relevant documentation was found for "
                "the requested Pydantic version."
            ),
            "chunk_ids": [],
        }

    top_chunk = retrieval_results.iloc[0]

    supporting_chunks = list(
        retrieval_results["chunk_id"]
    )

    answer = (
        f"Requested Pydantic version: {requested_version}\n\n"
        f"The highest-ranked documentation suggests:\n\n"
        f"{top_chunk['chunk_text'][:900]}\n\n"
        f"This answer was generated using only "
        f"{requested_version} documentation."
    )

    return {
        "requested_version": requested_version,
        "query": query,
        "answer": answer,
        "chunk_ids": supporting_chunks,
        "primary_document": top_chunk["document_id"],
        "primary_section": top_chunk["section_title"],
    }


versionguard_response = generate_answer(
    query=demo_query,
    requested_version=requested_version,
    retrieval_results=aware_results,
)

print("=" * 80)
print("VersionGuard Response")
print("=" * 80)

print(f"\nQuery:\n{versionguard_response['query']}\n")

print(versionguard_response["answer"])

print("\nSupporting Chunk IDs:")

for chunk_id in versionguard_response["chunk_ids"]:
    print(f"- {chunk_id}")

print("\nPrimary Source:")
print(versionguard_response["primary_document"])

print("\nSection:")
print(versionguard_response["primary_section"])

VersionGuard Response

Query:
How do I create a validator?

Requested Pydantic version: v1

The highest-ranked documentation suggests:

Validators can do a few more complex things:

{!.tmp_examples/validators_pre_item.md!}

A few more things to note:

* a single validator can be applied to multiple fields by passing it multiple field names
* a single validator can also be called on *all* fields by passing the special value `'*'`
* the keyword argument `pre` will cause the validator to be called prior to other validation
* passing `each_item=True` will result in the validator being applied to individual values
  (e.g. of `List`, `Dict`, `Set`, etc.), rather than the whole object

This answer was generated using only v1 documentation.

Supporting Chunk IDs:
- pydantic-v1-validators-chunk-0003
- pydantic-v1-model_config-chunk-0008
- pydantic-v1-validators-chunk-0004
- pydantic-v1-validators-chunk-0008
- pydantic-v1-models-chunk-0015

Primary Source:
pydantic-v1-validators

Section:
Pre an

## Current Pipeline Interpretation

The accepted Pydantic documents have successfully progressed through ingestion, validation, processing, and chunk validation to become trustworthy model-facing assets for the VersionGuard retrieval system.

Quarantined records and chunks are preserved as evidence but are excluded from later processing stages until their validation issues are corrected. This prevents incomplete or inconsistent data from entering the retrieval pipeline while maintaining a record for future investigation.

The current prototype now provides the following information to support lineage, reproducibility, and debugging:

- Exact Pydantic source version
- Original source URL
- Documentation URL
- Capture timestamp
- HTTP result
- Document identifier
- Chunk identifier
- Document and chunk SHA-256 content hashes
- Character counts
- Validation status
- Validation errors
- Ingestion run identifier
- Retrieval asset version
- Prompt template version
- Retrieved chunk identifiers
- Requested Pydantic version
- Version-aware retrieval results

This prototype demonstrates how preserving version metadata throughout the pipeline helps reduce quiet failures by ensuring that retrieval and response generation use documentation from the appropriate Pydantic version while maintaining evidence that supports future debugging and reproducibility.